# 9.3 激活修补 / Activation patching

在两次只差一处的运行之间交换内部激活，测量每个位置、每一层和每个注意力头对输出的因果作用，并用梯度做快速近似。

Swapping internal activations between two runs that differ in one place, to measure the causal effect of every position, layer and attention head on the output, with a fast approximation by gradients.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 修补残差流 / Patch the residual stream

实现保存激活的 `run_with_cache`、归一化度量，以及把干净运行中某一层某个位置的残差流换入损坏运行的 `patch_residual`，再对所有层和位置计算修补效应。数据是代码中给出的 8 对间接宾语识别句子。测试检查缓存与模型实际计算的一致、修补一层的全部位置可以完全恢复干净运行、修补两次运行相同的位置没有效果，以及效应在层和位置上的分布。运行时间约 35 秒。

Implement `run_with_cache`, which stores activations, the normalized metric, and `patch_residual`, which moves the residual stream of one layer and position from the clean run into the corrupted run, and then compute the patching effect for all layers and positions. The data are the 8 pairs of indirect object identification sentences given in the code. The tests check that the cache agrees with what the model computes, that patching all positions of a layer fully restores the clean run, that patching a position at which the two runs agree has no effect, and how the effect is distributed over layers and positions. The run takes about 35 seconds.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_gpt2():
    tok = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
    return model, tok


PAIRS = [(" Mary", " John"), (" Tom", " James"), (" Sarah", " David"), (" Paul", " Emma"),
         (" John", " Mary"), (" James", " Tom"), (" David", " Sarah"), (" Emma", " Paul")]
TEMPLATE = "When{a} and{b} went to the store,{s} gave a drink to"
S2, END = 9, 13          # positions of the repeated name and of the last token


def ioi_batch(tok):
    """Eight prompt pairs of the indirect object identification task, all 14 tokens long.

    clean:   When A and B went to the store, B gave a drink to   -> A
    corrupt: When A and B went to the store, A gave a drink to   -> B
    The two differ at position S2 only. Returns (clean, corrupt, ans, wrong): ids (8, 14) twice and
    the token ids of A and B, each (8,).
    """
    enc = lambda s: tok(s, return_tensors="pt").input_ids[0]
    clean = torch.stack([enc(TEMPLATE.format(a=a, b=b, s=b)) for a, b in PAIRS])
    corrupt = torch.stack([enc(TEMPLATE.format(a=a, b=b, s=a)) for a, b in PAIRS])
    ans = torch.tensor([tok(a).input_ids[0] for a, b in PAIRS])
    wrong = torch.tensor([tok(b).input_ids[0] for a, b in PAIRS])
    return clean, corrupt, ans, wrong


def logit_diff(logits, ans, wrong):
    """logit(ans) - logit(wrong) at the last position. logits (N, T, V) -> (N,)."""
    # TODO 1: take the last position, then gather the logits of ans and of wrong and subtract
    raise NotImplementedError


def block_output(out):
    return out[0] if isinstance(out, tuple) else out


@torch.no_grad()
def run_with_cache(model, ids):
    """Run the model and store activations. Returns (logits, cache).

    cache[("resid", l)]: output of block l, shape (N, T, 768)
    cache[("z", l)]:     input of attn.c_proj of block l, the concatenated head outputs, (N, T, 768)
    """
    cache, handles = {}, []
    # TODO 2: for every block l register
    #   a forward hook on the block that stores block_output(out).clone() under ("resid", l)
    #   a forward pre-hook on block.attn.c_proj that stores args[0].clone() under ("z", l)
    #   Bind l in the lambda with a default argument (lambda m, a, out, l=l: ...).
    # TODO 3: run the model, remove the handles in a finally clause, return (logits, cache)
    raise NotImplementedError


def normalized(patched, clean, corrupt):
    """0 if the patched run behaves like the corrupted run, 1 if like the clean run. Tensors (N,) -> float."""
    # TODO 4: (mean patched - mean corrupt) / (mean clean - mean corrupt), as a Python float
    raise NotImplementedError


@torch.no_grad()
def patch_residual(model, corrupt_ids, clean_cache, layer, pos):
    """Run on the corrupted input with the output of block `layer` at position `pos` replaced by its
    clean value. pos is an int or slice(None). Returns the logits (N, T, V)."""
    # TODO 5: a forward hook on model.transformer.h[layer] that copies block_output(out), overwrites
    #         [:, pos] with the clean value and returns the result. If out is a tuple, return a tuple
    #         with the new tensor in first place: (x,) + tuple(out[1:]).
    # TODO 6: run on corrupt_ids, remove the handle, return the logits
    raise NotImplementedError


def residual_patching_map(model, clean_ids, corrupt_ids, ans, wrong):
    """Normalized effect of patching every (layer, position). Returns a tensor (n_layer, T)."""
    # TODO 7: cache the clean run, compute the logit differences of the clean and the corrupted run,
    #         then loop over layers and positions and fill a tensor (n_layer, T) with normalized(...)
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch

model, tok = load_gpt2()
clean, corrupt, ans, wrong = ioi_batch(tok)
assert clean.shape == corrupt.shape == (8, 14)
assert (clean != corrupt).sum(0).tolist() == [0] * 9 + [8] + [0] * 4, "the pairs differ at position 9 only"

# 1. Logit difference on a hand-made tensor.
fake = torch.zeros(2, 3, 5); fake[0, -1] = torch.tensor([1.0, 4.0, 0, 0, 0]); fake[1, -1] = torch.tensor([0, 0, 2.0, 0, 7.0])
assert logit_diff(fake, torch.tensor([1, 2]), torch.tensor([0, 4])).tolist() == [3.0, -5.0]
assert abs(normalized(torch.tensor([1.0, 3.0]), torch.tensor([4.0, 4.0]), torch.tensor([0.0, 0.0])) - 0.5) < 1e-6

# 2. The cache holds what the model computes, and no hook is left behind.
hooks_now = lambda: sum(len(b._forward_hooks) + len(b.attn.c_proj._forward_pre_hooks) for b in model.transformer.h)
clean_logits, cache = run_with_cache(model, clean)
assert hooks_now() == 0, "remove the hooks after the run"
with torch.no_grad():
    ref = model(clean, output_hidden_states=True)
assert torch.allclose(clean_logits, ref.logits, atol=1e-4)
assert len(cache) == 24 and cache[("resid", 5)].shape == (8, 14, 768) and cache[("z", 5)].shape == (8, 14, 768)
for l in range(11):
    assert torch.allclose(cache[("resid", l)], ref.hidden_states[l + 1], rtol=1e-4, atol=1e-2), l
n_hooks = hooks_now()          # transformers itself may keep hooks on the blocks after output_hidden_states

# 3. The model solves the task on the clean prompts and is misled on the corrupted ones.
with torch.no_grad():
    corrupt_logits = model(corrupt).logits
c, x = logit_diff(clean_logits, ans, wrong), logit_diff(corrupt_logits, ans, wrong)
assert (c > 0).all() and (x < 0).all()

# 4. Patching all positions of a layer turns the corrupted run into the clean run.
full = patch_residual(model, corrupt, cache, 6, slice(None))
assert torch.allclose(full, clean_logits, atol=1e-3)
assert hooks_now() == n_hooks
# Patching a position at which the two inputs agree changes nothing.
same = patch_residual(model, corrupt, cache, 6, 3)
assert torch.allclose(same, corrupt_logits, atol=1e-4)

# 5. The full map.
M = residual_patching_map(model, clean, corrupt, ans, wrong)
assert M.shape == (12, 14)
assert M[:, :S2].abs().max() < 1e-3, "positions before the changed token carry no difference"
assert M[0, S2] > 0.95, "in layer 0 the whole difference sits at the changed token"
assert abs(M[11, END] - 1) < 1e-3, "after the last block only the last position matters"
assert M[11, S2] < 0.01
cross = min(l for l in range(12) if M[l, END] > M[l, S2])
assert 6 <= cross <= 9, cross

print(f"logit difference: clean {c.mean():.2f}, corrupted {x.mean():.2f} (mean over 8 pairs)")
tokens = [tok.decode(t) for t in clean[0]]
print("normalized effect of patching the residual stream after each block, at the positions from S2 on")
print("layer  " + "".join(f"{t!r:>9}" for t in tokens[S2:]))
for l in range(12):
    print(f"{l:>5}  " + "".join(f"{M[l, p].item():>9.2f}" for p in range(S2, 14)))
print(f"the information moves from position {S2} to the last position between layers {cross - 1} and {cross}")

<details><summary><b>提示 1 / Hint 1</b></summary>

在循环中注册钩子时，用默认参数把当前的层号固定下来，写成 `lambda m, a, out, l=l: ...`。

When registering hooks in a loop, fix the current layer number with a default argument, as in `lambda m, a, out, l=l: ...`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

前向钩子返回一个值时，这个值会替换模块的输出。块的输出是元组时，需要返回同样结构的元组，`(x,) + tuple(out[1:])`。

When a forward hook returns a value, the value replaces the output of the module. If the output of a block is a tuple, a tuple of the same structure has to be returned, `(x,) + tuple(out[1:])`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

`x[:, pos] = clean_cache[('resid', layer)][:, pos]` 对整数位置和 `slice(None)` 都适用。修改之前先 `clone()`。

`x[:, pos] = clean_cache[('resid', layer)][:, pos]` works for an integer position and for `slice(None)` alike. Call `clone()` before modifying.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_gpt2():
    tok = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
    return model, tok


PAIRS = [(" Mary", " John"), (" Tom", " James"), (" Sarah", " David"), (" Paul", " Emma"),
         (" John", " Mary"), (" James", " Tom"), (" David", " Sarah"), (" Emma", " Paul")]
TEMPLATE = "When{a} and{b} went to the store,{s} gave a drink to"
S2, END = 9, 13          # positions of the repeated name and of the last token


def ioi_batch(tok):
    """Eight prompt pairs of the indirect object identification task, all 14 tokens long.

    clean:   When A and B went to the store, B gave a drink to   -> A
    corrupt: When A and B went to the store, A gave a drink to   -> B
    The two differ at position S2 only. Returns (clean, corrupt, ans, wrong): ids (8, 14) twice and
    the token ids of A and B, each (8,).
    """
    enc = lambda s: tok(s, return_tensors="pt").input_ids[0]
    clean = torch.stack([enc(TEMPLATE.format(a=a, b=b, s=b)) for a, b in PAIRS])
    corrupt = torch.stack([enc(TEMPLATE.format(a=a, b=b, s=a)) for a, b in PAIRS])
    ans = torch.tensor([tok(a).input_ids[0] for a, b in PAIRS])
    wrong = torch.tensor([tok(b).input_ids[0] for a, b in PAIRS])
    return clean, corrupt, ans, wrong


def logit_diff(logits, ans, wrong):
    """logit(ans) - logit(wrong) at the last position. logits (N, T, V) -> (N,)."""
    last = logits[:, -1]
    return (last.gather(1, ans[:, None]) - last.gather(1, wrong[:, None]))[:, 0]


def block_output(out):
    return out[0] if isinstance(out, tuple) else out


@torch.no_grad()
def run_with_cache(model, ids):
    """Run the model and store activations. Returns (logits, cache).

    cache[("resid", l)]: output of block l, shape (N, T, 768)
    cache[("z", l)]:     input of attn.c_proj of block l, the concatenated head outputs, (N, T, 768)
    """
    cache, handles = {}, []
    for l, block in enumerate(model.transformer.h):
        handles.append(block.register_forward_hook(
            lambda m, a, out, l=l: cache.__setitem__(("resid", l), block_output(out).clone())))
        handles.append(block.attn.c_proj.register_forward_pre_hook(
            lambda m, a, l=l: cache.__setitem__(("z", l), a[0].clone())))
    try:
        logits = model(ids).logits
    finally:
        for h in handles:
            h.remove()
    return logits, cache


def normalized(patched, clean, corrupt):
    """0 if the patched run behaves like the corrupted run, 1 if like the clean run. Tensors (N,) -> float."""
    return ((patched.mean() - corrupt.mean()) / (clean.mean() - corrupt.mean())).item()


@torch.no_grad()
def patch_residual(model, corrupt_ids, clean_cache, layer, pos):
    """Run on the corrupted input with the output of block `layer` at position `pos` replaced by its
    clean value. pos is an int or slice(None). Returns the logits (N, T, V)."""
    def hook(module, args, out):
        x = block_output(out).clone()
        x[:, pos] = clean_cache[("resid", layer)][:, pos]
        return (x,) + tuple(out[1:]) if isinstance(out, tuple) else x
    handle = model.transformer.h[layer].register_forward_hook(hook)
    try:
        return model(corrupt_ids).logits
    finally:
        handle.remove()


def residual_patching_map(model, clean_ids, corrupt_ids, ans, wrong):
    """Normalized effect of patching every (layer, position). Returns a tensor (n_layer, T)."""
    clean_logits, cache = run_with_cache(model, clean_ids)
    with torch.no_grad():
        corrupt_logits = model(corrupt_ids).logits
    c, x = logit_diff(clean_logits, ans, wrong), logit_diff(corrupt_logits, ans, wrong)
    L, T = model.config.n_layer, clean_ids.shape[1]
    out = torch.zeros(L, T)
    for l in range(L):
        for p in range(T):
            out[l, p] = normalized(logit_diff(patch_residual(model, corrupt_ids, cache, l, p), ans, wrong), c, x)
    return out
```

</details>

## 2. 修补注意力头 / Patch attention heads

实现对单个注意力头输出的修补，可以限定在某一个位置上，并计算 144 个头的修补效应。测试检查同一层的 12 个头全部修补等于替换整层的头输出、第 8 层第 10 个头只在最后一个位置起作用，以及效应最强的头与 Wang 等人描述的回路相符。请阅读打印出的表，注意哪些头在 S2 位置起作用，哪些在最后一个位置起作用。运行时间约 45 秒。

Implement patching of the output of a single attention head, optionally restricted to one position, and compute the patching effect of the 144 heads. The tests check that patching all 12 heads of a layer equals replacing the head outputs of the whole layer, that head 10 of layer 8 acts at the last position only, and that the heads with the strongest effect agree with the circuit described by Wang et al. Read the printed table and note which heads act at position S2 and which at the last position. The run takes about 45 seconds.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch


@torch.no_grad()
def patch_head(model, corrupt_ids, clean_cache, layer, head, pos=slice(None)):
    """Run on the corrupted input with the output of one head replaced by its clean value.

    The head output is the slice [head * 64, (head + 1) * 64) of the input of attn.c_proj.
    pos is an int or slice(None). Returns the logits (N, T, V).
    """
    d = model.config.n_embd // model.config.n_head
    sl = slice(head * d, (head + 1) * d)
    # TODO 1: a forward pre-hook on model.transformer.h[layer].attn.c_proj that clones args[0],
    #         overwrites z[:, pos, sl] with clean_cache[("z", layer)][:, pos, sl] and returns (z,)
    # TODO 2: run on corrupt_ids, remove the handle, return the logits
    raise NotImplementedError


def head_patching_map(model, clean_ids, corrupt_ids, ans, wrong, pos=slice(None)):
    """Normalized effect of patching every head. Returns a tensor (n_layer, n_head)."""
    # TODO 3: as residual_patching_map, with the loop over layers and heads
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch

model, tok = load_gpt2()
clean, corrupt, ans, wrong = ioi_batch(tok)
clean_logits, cache = run_with_cache(model, clean)
with torch.no_grad():
    corrupt_logits = model(corrupt).logits

# 1. Patching a head changes exactly its slice.
out = patch_head(model, corrupt, cache, 8, 10)
assert out.shape == corrupt_logits.shape
assert not torch.allclose(out, corrupt_logits, atol=1e-3)
assert sum(len(b.attn.c_proj._forward_pre_hooks) for b in model.transformer.h) == 0

# 2. Patching the 12 heads of a layer one after the other in the same run equals replacing the whole input of c_proj.
def patch_layer(layer):
    def hook(module, args): return (cache[("z", layer)].clone(),)
    h = model.transformer.h[layer].attn.c_proj.register_forward_pre_hook(hook)
    try:
        with torch.no_grad(): return model(corrupt).logits
    finally: h.remove()
hs = []
for hd in range(12):
    dd = 64
    def hook(module, args, hd=hd):
        z = args[0].clone(); z[:, :, hd * dd:(hd + 1) * dd] = cache[("z", 9)][:, :, hd * dd:(hd + 1) * dd]; return (z,)
    hs.append(model.transformer.h[9].attn.c_proj.register_forward_pre_hook(hook))
with torch.no_grad(): all12 = model(corrupt).logits
for h in hs: h.remove()
assert torch.allclose(all12, patch_layer(9), atol=1e-4)

# 3. Restricting the patch to a position.
c, x = logit_diff(clean_logits, ans, wrong), logit_diff(corrupt_logits, ans, wrong)
eff = lambda logits: normalized(logit_diff(logits, ans, wrong), c, x)
e_all, e_end, e_s2 = eff(patch_head(model, corrupt, cache, 8, 10)), eff(patch_head(model, corrupt, cache, 8, 10, END)), eff(patch_head(model, corrupt, cache, 8, 10, S2))
assert abs(e_all - e_end) < 0.02 and abs(e_s2) < 0.02, "head 8.10 acts at the last position"

# 4. The full map (144 forward passes).
M = head_patching_map(model, clean, corrupt, ans, wrong)
assert M.shape == (12, 12)
flat = M.flatten()
top = [(i // 12, i % 12) for i in flat.argsort(descending=True)[:8].tolist()]
neg = [(i // 12, i % 12) for i in flat.argsort()[:3].tolist()]
assert top[0] == (8, 10), top
assert set(neg[:2]) == {(10, 7), (11, 10)} and M[10, 7] < -0.1, "the two negative name movers work against the answer"
assert M[:3].abs().max() < 0.1

ROLE = {(2, 2): "previous token", (4, 11): "previous token", (0, 1): "duplicate token", (3, 0): "duplicate token", (0, 10): "duplicate token",
        (5, 5): "induction", (6, 9): "induction", (5, 8): "induction", (5, 9): "induction",
        (7, 3): "S-inhibition", (7, 9): "S-inhibition", (8, 6): "S-inhibition", (8, 10): "S-inhibition",
        (9, 9): "name mover", (9, 6): "name mover", (10, 0): "name mover",
        (10, 7): "negative name mover", (11, 10): "negative name mover",
        (10, 10): "backup name mover", (10, 6): "backup name mover", (10, 2): "backup name mover", (10, 1): "backup name mover",
        (11, 2): "backup name mover", (9, 7): "backup name mover", (9, 0): "backup name mover", (11, 9): "backup name mover"}
listed = sum((l, h) in ROLE for l, h in top)
assert listed >= 6, "most of the strongest heads belong to the circuit described by Wang et al."
print("head   effect   at END   at S2   role in Wang et al. (2022)")
for l, h in top + neg:
    at_end, at_s2 = eff(patch_head(model, corrupt, cache, l, h, END)), eff(patch_head(model, corrupt, cache, l, h, S2))
    print(f"{l}.{h:<4} {M[l, h].item():>+6.2f}   {at_end:>+6.2f}  {at_s2:>+6.2f}   {ROLE.get((l, h), 'not listed')}")
print(f"heads with |effect| > 0.05: {(M.abs() > 0.05).sum().item()} of 144")
print(f"sum over all heads {M.sum().item():.2f}")
name_movers = [(9, 9), (9, 6), (10, 0)]
print("name movers of Wang et al.: " + ", ".join(f"{l}.{h} {M[l, h].item():+.2f}" for l, h in name_movers))

<details><summary><b>提示 1 / Hint 1</b></summary>

头 $h$ 的输出是 c_proj 输入的切片 `[h * 64, (h + 1) * 64)`，与 9.1 的消融相同，只是把置零改为换成干净运行的值。

The output of head $h$ is the slice `[h * 64, (h + 1) * 64)` of the input of c_proj, as in the ablation of 9.1, with zeroing replaced by the value of the clean run.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`z[:, pos, sl] = clean_cache[('z', layer)][:, pos, sl]`，其中 `sl = slice(head * d, (head + 1) * d)`。

`z[:, pos, sl] = clean_cache[('z', layer)][:, pos, sl]` with `sl = slice(head * d, (head + 1) * d)`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch


@torch.no_grad()
def patch_head(model, corrupt_ids, clean_cache, layer, head, pos=slice(None)):
    """Run on the corrupted input with the output of one head replaced by its clean value.

    The head output is the slice [head * 64, (head + 1) * 64) of the input of attn.c_proj.
    pos is an int or slice(None). Returns the logits (N, T, V).
    """
    d = model.config.n_embd // model.config.n_head
    sl = slice(head * d, (head + 1) * d)

    def hook(module, args):
        z = args[0].clone()
        z[:, pos, sl] = clean_cache[("z", layer)][:, pos, sl]
        return (z,)
    handle = model.transformer.h[layer].attn.c_proj.register_forward_pre_hook(hook)
    try:
        return model(corrupt_ids).logits
    finally:
        handle.remove()


def head_patching_map(model, clean_ids, corrupt_ids, ans, wrong, pos=slice(None)):
    """Normalized effect of patching every head. Returns a tensor (n_layer, n_head)."""
    clean_logits, cache = run_with_cache(model, clean_ids)
    with torch.no_grad():
        corrupt_logits = model(corrupt_ids).logits
    c, x = logit_diff(clean_logits, ans, wrong), logit_diff(corrupt_logits, ans, wrong)
    L, H = model.config.n_layer, model.config.n_head
    out = torch.zeros(L, H)
    for l in range(L):
        for h in range(H):
            out[l, h] = normalized(logit_diff(patch_head(model, corrupt_ids, cache, l, h, pos), ans, wrong), c, x)
    return out
```

</details>

## 3. 归因修补 / Attribution patching

在损坏运行上做一次前向和一次反向计算，取得度量对各层头输出和残差流的梯度，再按推导 2 估计所有头和所有残差流位置的修补效应。测试先用有限差分检查梯度是正确的，再把估计值与实际修补的结果比较。请查看打印出的对比，注意估计在哪里准确、在哪里偏离。运行时间约 70 秒，其中大部分用于实际修补。

Run one forward and one backward pass on the corrupted input to obtain the gradient of the metric with respect to the head outputs and the residual stream of every layer, and estimate following derivation 2 the patching effect of all heads and of all residual stream positions. The tests check the gradient by a finite difference and then compare the estimate with the result of real patching. Look at the printed comparison and note where the estimate is accurate and where it deviates. The run takes about 70 seconds, most of it for the real patching.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch


def run_with_grad(model, ids, ans, wrong):
    """Run on ids and return (activations, gradients) of the mean logit difference.

    Both are dicts with the keys ("z", l) and ("resid", l) as in run_with_cache, the gradients having
    the same shapes as the activations.
    """
    acts, handles = {}, []

    def keep(key, t):
        t.retain_grad()          # ask autograd to keep the gradient of this intermediate tensor
        acts[key] = t
    # TODO 1: register the same two hooks per block as in run_with_cache, calling keep(...) on the
    #         tensors themselves (no clone, no torch.no_grad)
    # TODO 2: metric = mean logit difference of model(ids).logits, then metric.backward(), remove the handles
    # TODO 3: return ({key: activation.detach()}, {key: activation.grad.detach()})
    raise NotImplementedError


def attribution_heads(model, clean_cache, corrupt_acts, corrupt_grads, scale):
    """First-order estimate of the effect of patching every head at all positions.

    estimate = sum over batch, positions and the 64 dimensions of (clean - corrupt) * gradient.
    The gradient belongs to the mean over the batch, so the sum over the batch gives the change of
    the mean. scale is mean clean logit difference minus mean corrupted one. Returns (n_layer, n_head).
    """
    # TODO 4: for every layer form the product, reshape (N, T, 768) to (N, T, 12, 64) and sum over
    #         all axes except the head axis, then divide by scale
    raise NotImplementedError


def attribution_residual(model, clean_cache, corrupt_acts, corrupt_grads, scale):
    """The same estimate for the residual stream at every (layer, position). Returns (n_layer, T)."""
    # TODO 5: the product for ("resid", l), summed over the batch and the 768 dimensions
    raise NotImplementedError

In [ ]:
# 检查 / checks
import time
import torch

model, tok = load_gpt2()
clean, corrupt, ans, wrong = ioi_batch(tok)
clean_logits, cache = run_with_cache(model, clean)

t0 = time.time()
acts, grads = run_with_grad(model, corrupt, ans, wrong)
t_attr = time.time() - t0

# 1. Activations equal those of an ordinary run, gradients have the same shapes.
corrupt_logits, corrupt_cache = run_with_cache(model, corrupt)
assert set(acts) == set(corrupt_cache) == set(grads)
for k in acts:
    assert torch.allclose(acts[k], corrupt_cache[k], rtol=1e-4, atol=1e-3), k
    assert grads[k].shape == acts[k].shape and not acts[k].requires_grad
assert sum(len(b.attn.c_proj._forward_pre_hooks) for b in model.transformer.h) == 0

# 2. The gradient is a real derivative: a small step along it changes the metric as predicted.
c, x = logit_diff(clean_logits, ans, wrong), logit_diff(corrupt_logits, ans, wrong)
g = grads[("resid", 8)]
eps = 1e-3 / g.norm()
def hook(module, args, out):
    y = block_output(out) + eps * g
    return (y,) + tuple(out[1:]) if isinstance(out, tuple) else y
h = model.transformer.h[8].register_forward_hook(hook)
with torch.no_grad():
    moved = logit_diff(model(corrupt).logits, ans, wrong).mean().item()
h.remove()
predicted = (eps * g.pow(2).sum()).item()
assert abs((moved - x.mean().item()) - predicted) < 0.05 * abs(predicted) + 1e-5, (moved - x.mean().item(), predicted)

# 3. Heads: the estimate against real patching.
scale = (c.mean() - x.mean()).item()
A = attribution_heads(model, cache, acts, grads, scale)
assert A.shape == (12, 12)
t0 = time.time()
M = head_patching_map(model, clean, corrupt, ans, wrong)
t_patch = time.time() - t0
r = torch.corrcoef(torch.stack([A.flatten(), M.flatten()]))[0, 1].item()
assert r > 0.9, r
top_true = set(M.flatten().abs().argsort(descending=True)[:8].tolist())
top_est = set(A.flatten().abs().argsort(descending=True)[:8].tolist())
assert len(top_true & top_est) >= 6

# 4. Residual stream: the estimate against real patching.
AR_ = attribution_residual(model, cache, acts, grads, scale)
MR = residual_patching_map(model, clean, corrupt, ans, wrong)
assert AR_.shape == MR.shape == (12, 14)
assert AR_[:, :S2].abs().max() < 1e-4, "no difference, no attribution"

print(f"heads: one forward and one backward pass {t_attr:.1f} s, 144 patching runs {t_patch:.1f} s")
print(f"correlation between estimate and real effect over the 144 heads: {r:.3f}, {len(top_true & top_est)} of the 8 strongest heads agree")
worst = (A - M).abs().flatten().argsort(descending=True)[:4].tolist()
print("largest errors: " + ", ".join(f"{i // 12}.{i % 12} estimate {A.flatten()[i]:+.2f} real {M.flatten()[i]:+.2f}" for i in worst))
print("residual stream at S2 and END, estimate / real")
for l in range(12):
    print(f"  layer {l:>2}   S2 {AR_[l, S2].item():>+6.2f} / {MR[l, S2].item():>+5.2f}    END {AR_[l, END].item():>+6.2f} / {MR[l, END].item():>+5.2f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

中间张量默认不保留梯度。在钩子里对它调用 `retain_grad()`，反向传播后就可以从 `.grad` 读取。这里不能使用 `torch.no_grad()`，也不要 `clone()`。

Intermediate tensors do not keep their gradients by default. Calling `retain_grad()` on them inside the hook makes the gradient available in `.grad` after the backward pass. `torch.no_grad()` cannot be used here, and `clone()` should be avoided.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

度量取 8 对句子的平均 logit 差，因此梯度已经带有 $1/8$ 的系数，对批次求和得到的就是平均值的变化量。

The metric is the mean logit difference over the 8 pairs, so the gradient already carries a factor of $1/8$, and summing over the batch gives the change of the mean.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

`d.reshape(N, T, 12, 64).sum((0, 1, 3))` 得到每个头的估计值。

`d.reshape(N, T, 12, 64).sum((0, 1, 3))` gives the estimate for every head.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch


def run_with_grad(model, ids, ans, wrong):
    """Run on ids and return (activations, gradients) of the mean logit difference.

    Both are dicts with the keys ("z", l) and ("resid", l) as in run_with_cache, the gradients having
    the same shapes as the activations.
    """
    acts, handles = {}, []

    def keep(key, t):
        t.retain_grad()
        acts[key] = t
    for l, block in enumerate(model.transformer.h):
        handles.append(block.register_forward_hook(lambda m, a, out, l=l: keep(("resid", l), block_output(out))))
        handles.append(block.attn.c_proj.register_forward_pre_hook(lambda m, a, l=l: keep(("z", l), a[0])))
    try:
        metric = logit_diff(model(ids).logits, ans, wrong).mean()
        model.zero_grad()
        metric.backward()
    finally:
        for h in handles:
            h.remove()
    grads = {k: v.grad.detach() for k, v in acts.items()}
    return {k: v.detach() for k, v in acts.items()}, grads


def attribution_heads(model, clean_cache, corrupt_acts, corrupt_grads, scale):
    """First-order estimate of the effect of patching every head at all positions.

    estimate = sum over batch, positions and the 64 dimensions of (clean - corrupt) * gradient.
    The gradient belongs to the mean over the batch, so the sum over the batch gives the change of
    the mean. scale is mean clean logit difference minus mean corrupted one. Returns (n_layer, n_head).
    """
    L, H = model.config.n_layer, model.config.n_head
    out = torch.zeros(L, H)
    for l in range(L):
        d = (clean_cache[("z", l)] - corrupt_acts[("z", l)]) * corrupt_grads[("z", l)]
        N, T, D = d.shape
        out[l] = d.reshape(N, T, H, D // H).sum((0, 1, 3)) / scale
    return out


def attribution_residual(model, clean_cache, corrupt_acts, corrupt_grads, scale):
    """The same estimate for the residual stream at every (layer, position). Returns (n_layer, T)."""
    L = model.config.n_layer
    rows = []
    for l in range(L):
        d = (clean_cache[("resid", l)] - corrupt_acts[("resid", l)]) * corrupt_grads[("resid", l)]
        rows.append(d.sum((0, 2)) / scale)
    return torch.stack(rows)
```

</details>

## 4. 用 TransformerLens 复现并核对 / Reproduce and cross-check with TransformerLens

用 TransformerLens 4 的 `run_with_cache` 和 `run_with_hooks` 重写残差流修补，并与练习 1 的结果比较。需要先安装 `transformer_lens`。测试检查该库给出的 logits、残差流和头输出与 Hugging Face 模型一致，以及两种实现得到的修补结果相同。运行时间约两分钟。

Rewrite residual stream patching with `run_with_cache` and `run_with_hooks` of TransformerLens 4 and compare with the result of exercise 1. `transformer_lens` has to be installed first. The tests check that the logits, residual stream and head outputs given by the library agree with the Hugging Face model, and that the two implementations yield the same patching result. The run takes about two minutes.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch


def load_bridge():
    """GPT-2 small wrapped by TransformerLens 4. Requires `pip install transformer_lens`."""
    from transformer_lens import TransformerBridge
    return TransformerBridge.boot_transformers("gpt2")


@torch.no_grad()
def tl_residual_patching_map(bridge, clean_ids, corrupt_ids, ans, wrong):
    """The residual patching map of exercise 1, written with the TransformerLens interface."""
    # TODO 1: clean_logits, cache = bridge.run_with_cache(clean_ids). corrupt_logits = bridge(corrupt_ids)
    # TODO 2: the output of block l is stored under the name f"blocks.{l}.hook_resid_post"
    # TODO 3: a hook has the signature hook(act, hook) and returns the new activation. hook.name is the
    #         name it was attached to. Run with bridge.run_with_hooks(corrupt_ids, fwd_hooks=[(name, fn)]),
    #         which removes the hook again by itself.
    raise NotImplementedError

In [ ]:
# 检查 / checks
import time
import torch

model, tok = load_gpt2()
clean, corrupt, ans, wrong = ioi_batch(tok)
bridge = load_bridge()

# 1. The wrapped model computes the same logits and the same residual stream as the Hugging Face model.
tl_logits, tl_cache = bridge.run_with_cache(clean)
hf_logits, hf_cache = run_with_cache(model, clean)
assert torch.allclose(tl_logits, hf_logits, atol=1e-3)
for l in (0, 5, 11):
    assert torch.allclose(tl_cache[f"blocks.{l}.hook_resid_post"], hf_cache[("resid", l)], rtol=1e-4, atol=1e-2), l
z = tl_cache["blocks.8.attn.hook_z"]
assert z.shape == (8, 14, 12, 64), "TransformerLens stores the head outputs with a separate head axis"
assert torch.allclose(z.reshape(8, 14, 768), hf_cache[("z", 8)], rtol=1e-4, atol=1e-3)

# 2. The two implementations of the patching map agree.
t0 = time.time(); mine = residual_patching_map(model, clean, corrupt, ans, wrong); t_mine = time.time() - t0
t0 = time.time(); theirs = tl_residual_patching_map(bridge, clean, corrupt, ans, wrong); t_tl = time.time() - t0
assert theirs.shape == (12, 14)
assert torch.allclose(mine, theirs, atol=1e-2), (mine - theirs).abs().max()

print(f"largest difference between the two maps: {(mine - theirs).abs().max().item():.5f}")
print(f"raw hooks {t_mine:.1f} s, TransformerLens {t_tl:.1f} s")
print(f"number of hook points TransformerLens exposes on GPT-2 small: {len(tl_cache)}")
print("hook points of one block:", ", ".join(k.split('blocks.8.')[1] for k in tl_cache if k.startswith("blocks.8.hook_")))

<details><summary><b>提示 1 / Hint 1</b></summary>

钩子函数的签名是 `fn(act, hook)`，返回修改后的激活。`hook.name` 是它所挂位置的名字，可以直接用来从缓存中取对应的干净激活。

A hook function has the signature `fn(act, hook)` and returns the modified activation. `hook.name` is the name of the place it is attached to and can be used directly to fetch the matching clean activation from the cache.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`bridge.run_with_hooks(corrupt_ids, fwd_hooks=[(name, fn)])` 返回 logits，并在结束时自动移除钩子。

`bridge.run_with_hooks(corrupt_ids, fwd_hooks=[(name, fn)])` returns the logits and removes the hook automatically at the end.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch


def load_bridge():
    """GPT-2 small wrapped by TransformerLens 4. Requires `pip install transformer_lens`."""
    from transformer_lens import TransformerBridge
    return TransformerBridge.boot_transformers("gpt2")


@torch.no_grad()
def tl_residual_patching_map(bridge, clean_ids, corrupt_ids, ans, wrong):
    """The residual patching map of exercise 1, written with the TransformerLens interface."""
    clean_logits, cache = bridge.run_with_cache(clean_ids)
    corrupt_logits = bridge(corrupt_ids)
    c, x = logit_diff(clean_logits, ans, wrong), logit_diff(corrupt_logits, ans, wrong)
    L, T = bridge.cfg.n_layers, clean_ids.shape[1]
    out = torch.zeros(L, T)
    for l in range(L):
        name = f"blocks.{l}.hook_resid_post"
        for p in range(T):
            def hook(act, hook, p=p):
                act[:, p] = cache[hook.name][:, p]
                return act
            patched = bridge.run_with_hooks(corrupt_ids, fwd_hooks=[(name, hook)])
            out[l, p] = normalized(logit_diff(patched, ans, wrong), c, x)
    return out
```

</details>

## 5. 总效应、近似与实验设计 / Total effects, the approximation and experimental design

在纸上完成。
(a) 一个极简模型只有两个串联的组件，$a = f(x)$，$y = g(a)$，度量为 $m = y$。干净输入给出 $m = 1$，损坏输入给出 $m = 0$。分别修补 $a$ 和修补 $y$，各自的归一化效应是多少？它们的和说明了什么？
(b) 设度量对某个激活是线性的，$M(a) = c^\top a + c_0$。证明归因修补此时没有误差。
(c) 忽略偏置和缩放，LayerNorm 可写为 $\operatorname{LN}(x) = \sqrt{d}\,\dfrac{x - \mu(x)}{\lVert x - \mu(x) \rVert}$。设 $x$ 的均值为 0。证明 $\operatorname{LN}(x)$ 在 $x$ 方向上的方向导数为零，即 $\dfrac{d}{d\epsilon} \operatorname{LN}\big((1 + \epsilon) x\big) \Big|_{\epsilon = 0} = 0$，并说明这为什么会使残差流上的归因修补低估效应。
(d) 要研究模型在工具返回的结果与自身记忆冲突时依据哪一方作答。设计一组干净输入和损坏输入，说明它们在哪里不同、度量取什么，以及修补残差流后预期能够回答什么问题。再指出这一设计看不到哪些计算。

Work on paper.
(a) A minimal model has two components in series, $a = f(x)$ and $y = g(a)$, with the metric $m = y$. The clean input gives $m = 1$ and the corrupted input $m = 0$. What is the normalized effect of patching $a$ and of patching $y$? What does their sum show?
(b) Let the metric be linear in some activation, $M(a) = c^\top a + c_0$. Prove that attribution patching has no error in this case.
(c) Ignoring bias and scale, LayerNorm can be written as $\operatorname{LN}(x) = \sqrt{d}\,\dfrac{x - \mu(x)}{\lVert x - \mu(x) \rVert}$. Let $x$ have mean 0. Prove that the directional derivative of $\operatorname{LN}(x)$ along $x$ is zero, that is $\dfrac{d}{d\epsilon} \operatorname{LN}\big((1 + \epsilon) x\big) \Big|_{\epsilon = 0} = 0$, and explain why this makes attribution patching on the residual stream underestimate effects.
(d) The question is on which side a model relies when the result returned by a tool conflicts with its own memory. Design a set of clean and corrupted inputs, say where they differ and what the metric is, and state which question patching the residual stream is expected to answer. Then name the computations this design does not see.

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 修补 $a$ 之后，$y$ 会被重新计算。

(a) After $a$ is patched, $y$ is recomputed.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) $(1 + \epsilon) x$ 的均值仍为 0，范数为 $(1 + \epsilon) \lVert x \rVert$。

(c) $(1 + \epsilon) x$ still has mean 0, and its norm is $(1 + \epsilon) \lVert x \rVert$.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

(d) 可以参照间接宾语识别任务的做法，让两条输入只在一处不同。

(d) The indirect object identification task can serve as a model, with the two inputs differing in one place.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 修补 $a$，即把 $a$ 设为 $f(x_c)$，随后 $y = g(f(x_c)) = 1$，效应为 1。修补 $y$，直接把 $y$ 设为 1，效应也是 1。两者之和为 2。两个组件位于同一条通路上，修补其中任何一个都恢复了整条通路，所以效应之和把这条通路计算了两次。单个组件的修补效应是总效应，不是可以相加的份额。

(b) $M(a_c) - M(a_d) = c^\top (a_c - a_d)$。梯度为 $\nabla_a M = c$，与求值点无关。归因修补的估计是 $(a_c - a_d)^\top c$，与真实的差值完全相同。误差只来自 $M$ 的非线性。

(c) $x$ 的均值为 0 时，$\operatorname{LN}\big((1+\epsilon) x\big) = \sqrt{d}\, \dfrac{(1+\epsilon) x}{(1+\epsilon) \lVert x \rVert} = \sqrt{d}\, \dfrac{x}{\lVert x \rVert} = \operatorname{LN}(x)$，与 $\epsilon$ 无关，导数为零。LayerNorm 的输出不随输入的缩放而变化。残差流经过的每个子层都先做 LayerNorm，所以度量对残差流的梯度在 $x$ 方向上没有分量。干净与损坏的残差流之差 $a_c - a_d$ 中沿 $a_d$ 方向的那一部分与梯度的内积为零，被一阶近似完全忽略。实际修补时这一部分会改变归一化之后的向量，并不为零。差值越大、越接近整体替换，被忽略的部分越多。

(d) 一种设计如下。选取模型能够凭记忆答对的事实，例如某座建筑所在的城市。干净输入的上下文中包含一条给出另一个答案的工具结果，随后是问题。损坏输入与它相同，只是工具结果中的答案换成了第三个城市，或者换成与记忆一致的答案。两条输入只在工具结果中答案 token 的位置上不同。度量取“工具给出的答案”与“记忆中的答案”的 logit 差。对每一层、每个位置修补残差流，可以回答工具结果中的答案信息在哪些层停留在它原来的位置，在哪些层被搬到最后一个位置，以及在最后一个位置上从哪一层开始决定输出。进一步修补注意力头可以找出负责搬运的头。

这一设计看不到两条输入共有的计算，包括模型怎样从记忆中取出答案，以及怎样识别出上下文中有一条工具结果。要研究前者，需要另一组输入，使记忆中的答案不同而工具结果相同，例如替换问题中的主语。9.5 的综合项目将按这一思路展开。

---

(a) Patching $a$ sets $a$ to $f(x_c)$, after which $y = g(f(x_c)) = 1$, an effect of 1. Patching $y$ sets $y$ to 1 directly, again an effect of 1. The two sum to 2. The two components lie on one pathway, and patching either restores the whole pathway, so the sum counts the pathway twice. The patching effect of a single component is a total effect and not a share that can be added.

(b) $M(a_c) - M(a_d) = c^\top (a_c - a_d)$. The gradient is $\nabla_a M = c$, independent of the point of evaluation. The estimate of attribution patching is $(a_c - a_d)^\top c$, identical to the true difference. Errors come from the nonlinearity of $M$ alone.

(c) With $x$ of mean 0, $\operatorname{LN}\big((1+\epsilon) x\big) = \sqrt{d}\, \dfrac{(1+\epsilon) x}{(1+\epsilon) \lVert x \rVert} = \sqrt{d}\, \dfrac{x}{\lVert x \rVert} = \operatorname{LN}(x)$, independent of $\epsilon$, so the derivative is zero. The output of LayerNorm does not change with the scale of its input. Every sublayer the residual stream passes through applies LayerNorm first, so the gradient of the metric with respect to the residual stream has no component along $x$. The part of the difference $a_c - a_d$ between clean and corrupted residual stream that lies along $a_d$ has zero inner product with the gradient and is ignored entirely by the first-order approximation. In real patching this part changes the normalized vector and is not zero. The larger the difference and the closer to a wholesale replacement, the more is ignored.

(d) One design is as follows. Facts are chosen that the model answers correctly from memory, such as the city of some building. The context of the clean input contains a tool result that gives another answer, followed by the question. The corrupted input is the same except that the answer in the tool result is replaced by a third city, or by the answer that agrees with memory. The two inputs differ only at the position of the answer token in the tool result. The metric is the logit difference between the answer given by the tool and the answer from memory. Patching the residual stream for every layer and position answers in which layers the answer information of the tool result stays at its original position, in which layers it is carried to the last position, and from which layer on it decides the output at the last position. Patching attention heads further identifies the heads that do the carrying.

This design does not see the computation the two inputs share, including how the model retrieves the answer from memory and how it recognizes that the context contains a tool result. Studying the former takes another set of inputs in which the remembered answer differs while the tool result stays the same, for example by replacing the subject of the question. The capstone of 9.5 proceeds along these lines.

</details>